In [ ]:
from pathlib import Path
import importlib
import sys

# Resolve project root robustly whether notebook runs from project root or notebooks/
PROJECT_ROOT = (
    Path.cwd() if (Path.cwd() / "config" / "config.yaml").exists() else Path.cwd().parent
).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_module = importlib.import_module("data.load")
preprocess_module = importlib.import_module("data.preprocess")
health_index_module = importlib.import_module("features.health_index")
velocity_module = importlib.import_module("features.velocity")
variability_module = importlib.import_module("features.variability")

load_dataset = load_module.load_dataset
load_config = load_module.load_config
preprocess_train = preprocess_module.preprocess_train
preprocess_test = preprocess_module.preprocess_test
build_health_index = health_index_module.build_health_index
build_velocity = velocity_module.build_velocity
build_variability = variability_module.build_variability

config = load_config(PROJECT_ROOT / "config" / "config.yaml")
config["dataset"]["raw_path"] = str(PROJECT_ROOT / "data" / "raw")

train_raw, test_raw, _ = load_dataset(config)
train_proc, scaler, sensor_cols = preprocess_train(train_raw, config)
test_proc = preprocess_test(test_raw, config, scaler)

train_hi, test_hi, hi_artifacts = build_health_index(train_proc, test_proc, config)
train_vel, test_vel, velocity_artifacts = build_velocity(train_hi, test_hi, config)
train_final, test_final, var_artifacts = build_variability(train_vel, test_vel, config)

# Shape checks
assert train_final.shape[0] == 20631
assert test_final.shape[0] == 13096
assert "HI_velocity" in train_final.columns
assert "HI_variability" in train_final.columns

# Range checks
assert train_final["HI_variability"].between(0.0, 1.0).all()
assert test_final["HI_variability"].between(0.0, 1.0).all()

# No NaN checks
assert train_final["HI_velocity"].isna().sum() == 0
assert test_final["HI_velocity"].isna().sum() == 0
assert train_final["HI_variability"].isna().sum() == 0
assert test_final["HI_variability"].isna().sum() == 0

# Velocity direction: majority of late-life cycles should be negative
last_cycles = train_final.groupby("unit")["cycle"].max().reset_index(name="last_cycle")
train_with_last = train_final.merge(last_cycles, on="unit")
late_life = train_with_last[train_with_last["cycle"] > train_with_last["last_cycle"] * 0.75]
pct_negative = (late_life["HI_velocity"] < 0).mean()
assert pct_negative > 0.60, f"Expected >60% negative velocity in late life, got {pct_negative:.1%}"

# Variability: mean in last quarter of life should exceed first quarter
early = train_with_last[train_with_last["cycle"] <= train_with_last["last_cycle"] * 0.25]
late = train_with_last[train_with_last["cycle"] > train_with_last["last_cycle"] * 0.75]
assert late["HI_variability"].mean() > early["HI_variability"].mean(), (
    "Expected higher variability in late life than early life"
)

print("All checks passed")
print(f"  HI velocity - mean late life: {late_life['HI_velocity'].mean():.5f}")
print(f"  HI velocity - % negative late life: {pct_negative:.1%}")
print(f"  HI variability - mean early life: {early['HI_variability'].mean():.4f}")
print(f"  HI variability - mean late life: {late['HI_variability'].mean():.4f}")

[load] Train shape : (20631, 26)
[load] Test shape  : (13096, 26)
[load] RUL entries : 100
All checks passed
  HI velocity - mean late life: -0.00568
  HI velocity - % negative late life: 100.0%
  HI variability - mean early life: 0.2045
  HI variability - mean late life: 0.3340
